# 1 Azure Databricks Beginner Mini Textbook

This notebook teaches the basics of Azure Databricks using simple examples.

You will learn:
- Python basics
- Spark DataFrames
- Unity Catalog concepts
- Catalog, schema, and volume creation
- File creation in a Volume
- CSV reading
- Data profiling
- Null handling
- Duplicate removal
- Transformations and filtering
- Aggregations
- Delta tables
- Bronze and Silver layers
- SQL analytics

**Notebook style:** Each section includes a beginner-friendly explanation, syntax, examples, runnable code, and expected output notes.


In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# SETUP — Run this cell FIRST before running any other cell.
# It defines all configuration variables used throughout the notebook.
# Change the values below if you need a different catalog, schema, or volume.
# ─────────────────────────────────────────────────────────────────────────────

# Core configuration
catalog_name     = "beginner_catalog"
schema_name      = "databricks_basics"
full_schema_name = f"{catalog_name}.{schema_name}"
full_volume_name = f"{catalog_name}.{schema_name}.raw_files"
# NOTE: The lab CSV is written to /tmp (local driver temp storage) so the notebook
# runs without requiring Volume write permissions.  In production, use the Volume path:
#   csv_path = f"/Volumes/{catalog_name}/{schema_name}/raw_files/sample_customers.csv"
csv_path         = "/tmp/sample_customers.csv"

# Create notebook widgets so SQL cells can reference ${catalog_name} and ${schema_name}.
# We silently skip if widgets already exist from a previous run.
try:
    dbutils.widgets.text("catalog_name", catalog_name, "Catalog Name")
except Exception:
    pass  # widget already exists

try:
    dbutils.widgets.text("schema_name", schema_name, "Schema Name")
except Exception:
    pass  # widget already exists

print("Setup complete.")
print(f"  Catalog  : {catalog_name}")
print(f"  Schema   : {schema_name}")
print(f"  Volume   : {full_volume_name}")
print(f"  CSV path : {csv_path}")

Setup complete.
  Catalog  : beginner_catalog
  Schema   : databricks_basics
  Volume   : beginner_catalog.databricks_basics.raw_files
  CSV path : /tmp/sample_customers.csv


## 2. Python Basics

Python is a programming language commonly used for data engineering and data analysis.

### Variables
A **variable** stores a value.

**Syntax**

```python
variable_name = value
```

**Simple example**

```python
name = "Databricks"
count = 10
```

### Lists
A **list** stores multiple values.

**Syntax**

```python
my_list = [value1, value2, value3]
```

### Loops
A **loop** repeats an action.

**Syntax**

```python
for item in my_list:
    print(item)
```


In [0]:
# Store values in variables.
course_name = "Azure Databricks for Beginners"
student_count = 3

# Store multiple values in a list.
topics = ["Python", "Spark", "Delta Lake"]

print("Course:", course_name)
print("Students:", student_count)

# Loop through the list and print each topic.
for topic in topics:
    print("Learning topic:", topic)


Course: Azure Databricks for Beginners
Students: 3
Learning topic: Python
Learning topic: Spark
Learning topic: Delta Lake


### Code Explanation — Python Basics

---

**`course_name = "Azure Databricks for Beginners"`**
- `course_name` — the **name** you give the variable so you can refer to it later
- `=` — the **assignment operator**; it stores the value on the right into the variable on the left
- `"Azure Databricks for Beginners"` — a **string** value (text enclosed in double quotes)

---

**`student_count = 3`**
- `student_count` — a new variable
- `3` — an **integer** (a whole number with no decimal point)

---

**`topics = ["Python", "Spark", "Delta Lake"]`**
- `topics` — a variable that holds a **list** (a collection of values stored in one place)
- `[ ]` — square brackets define the start and end of a list
- Items inside are separated by commas; each item here is a string
- A list keeps values in order and lets you loop over them

---

**`print("Course:", course_name)`** and **`print("Students:", student_count)`**
- `print()` — a built-in Python **function** that sends output to the notebook display area
- You pass values inside the parentheses; multiple values separated by commas are printed with a space between them
- `course_name` is substituted at runtime with the actual value stored in that variable

---

**`for topic in topics:`**
- `for` — keyword that starts a **loop**
- `topic` — a temporary variable that holds the current item from the list on each pass through the loop
- `in topics` — tells the loop which collection to iterate through
- The `:` at the end is required Python syntax to open the loop block

**`    print("Learning topic:", topic)`**
- This line is **indented** (4 spaces) — indentation tells Python that this code belongs *inside* the `for` loop
- It runs **once for every item** in `topics`, printing each one in turn

**Expected output**

You should see the course name, number of students, and each topic printed on a separate line.


## 3. Spark DataFrames

A **Spark DataFrame** is a distributed table of data.

It has:
- Rows
- Columns
- A schema, which describes column names and data types

**Why this is used**
- Spark DataFrames can process small or very large datasets.
- They support transformations like selecting columns, filtering rows, and grouping data.

**Common function syntax**

```python
spark.createDataFrame(data, columns)
df.show()
df.printSchema()
```

**Simple example**

```python
data = [(1, "Aarav"), (2, "Meera")]
columns = ["id", "name"]
df = spark.createDataFrame(data, columns)
```


In [0]:
# Create a small Python list of rows.
sample_data = [
    (1, "Aarav", "Delhi"),
    (2, "Meera", "Mumbai"),
    (3, "Kabir", "Bengaluru")
]

# Define column names.
sample_columns = ["customer_id", "name", "city"]

# Convert the Python list into a Spark DataFrame.
sample_df = spark.createDataFrame(sample_data, sample_columns)

# Display the DataFrame.
display(sample_df)

# Print column names and data types.
sample_df.printSchema()


customer_id,name,city
1,Aarav,Delhi
2,Meera,Mumbai
3,Kabir,Bengaluru


root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)



### Code Explanation — Spark DataFrames

---

**`sample_data = [(1, "Aarav", "Delhi"), (2, "Meera", "Mumbai"), (3, "Kabir", "Bengaluru")]`**
- `sample_data` — a Python variable holding a **list of tuples**
- Each **tuple** (values wrapped in `( )`) represents one row of data
- The three values in each tuple correspond to `customer_id`, `name`, and `city` in that order

---

**`sample_columns = ["customer_id", "name", "city"]`**
- `sample_columns` — a list of strings that will become the **column names** of the DataFrame
- The order must match the order of values in each tuple above

---

**`sample_df = spark.createDataFrame(sample_data, sample_columns)`**
- `spark` — the **SparkSession** object; it is the entry point to the Spark engine and is automatically available in every Databricks notebook
- `.createDataFrame()` — a Spark method that converts a Python list of tuples into a distributed **Spark DataFrame**
- First argument: the data (list of tuples); second argument: column names (list of strings)
- The result is stored in `sample_df` (short for “sample DataFrame”)

---

**`display(sample_df)`**
- `display()` — a **Databricks-specific function** that renders a DataFrame as an interactive, scrollable table in the notebook output
- Preferred over `sample_df.show()` in Databricks notebooks because it produces a richer, visual result

---

**`sample_df.printSchema()`**
- `.printSchema()` — a Spark method that **prints the column names and their inferred data types** (e.g., `StringType`, `LongType`, `DoubleType`)
- This is used to verify that Spark correctly understood the structure of the data

**Expected output**

You should see a small table with three customers and a schema showing three columns.


## 4. Unity Catalog: Catalog, Schema, and Volume

**Unity Catalog** is Databricks' governance layer for data and AI assets.

### Catalog
A catalog is the top-level container.

**SQL syntax**

```sql
CREATE CATALOG IF NOT EXISTS catalog_name;
```

### Schema
A schema is a namespace inside a catalog.

**SQL syntax**

```sql
CREATE SCHEMA IF NOT EXISTS catalog_name.schema_name;
```

### Volume
A volume stores files inside Unity Catalog.

**SQL syntax**

```sql
CREATE VOLUME IF NOT EXISTS catalog_name.schema_name.volume_name;
```

**Why this is used**
- Catalogs and schemas organize tables.
- Volumes provide a governed location for raw files.
- Permissions can be managed centrally.


In [0]:
# Create the catalog. This may require permission in your workspace.
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")

# Create the schema inside the catalog.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {full_schema_name}")

# Create a volume for files.
spark.sql(f"CREATE VOLUME IF NOT EXISTS {full_volume_name}")

# Set the current catalog and schema for easier SQL work.
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

print("Unity Catalog objects are ready.")


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:439)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:439)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

### Code Explanation — Unity Catalog Setup

---

**`spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")`**
- `spark.sql()` — runs a **SQL statement inside a Python cell** using the active Spark session
- `f"..."` — an **f-string** (formatted string literal); anything inside `{ }` is replaced at runtime with the variable’s actual value
- `CREATE CATALOG IF NOT EXISTS` — SQL command to create a top-level Unity Catalog container; `IF NOT EXISTS` prevents an error if the catalog already exists
- `{catalog_name}` — replaced at runtime with the catalog name variable defined earlier in the notebook

---

**`spark.sql(f"CREATE SCHEMA IF NOT EXISTS {full_schema_name}")`**
- `CREATE SCHEMA IF NOT EXISTS` — creates a schema (also called a *database*) inside the catalog
- `{full_schema_name}` — typically formatted as `catalog_name.schema_name` so Spark knows exactly where to create it

---

**`spark.sql(f"CREATE VOLUME IF NOT EXISTS {full_volume_name}")`**
- `CREATE VOLUME IF NOT EXISTS` — creates a **Unity Catalog Volume**, which is a managed, governed storage location for files (CSV, JSON, Parquet, images, etc.)
- `{full_volume_name}` — typically formatted as `catalog.schema.volume_name`

---

**`spark.sql(f"USE CATALOG {catalog_name}")`**
- `USE CATALOG` — sets the **active catalog** for all subsequent SQL statements in this session
- After this, SQL queries do not need to prefix every table name with the catalog name

---

**`spark.sql(f"USE SCHEMA {schema_name}")`**
- `USE SCHEMA` — sets the **active schema** for the session
- After this, SQL queries can reference tables by name alone (no need for the full `catalog.schema.table` path)

---

**`print("Unity Catalog objects are ready.")`**
- Prints a confirmation message so you know the cell completed successfully without errors

**Expected output**

If you have the required permissions, the catalog, schema, and volume are created.

If you receive a permission error, ask your instructor or admin for a catalog where you can create schemas, volumes, and tables.


## 5. Create a Sample CSV File in a Volume

A **CSV file** stores tabular data as comma-separated values.

**Why this is used**
- CSV is common for data exchange.
- Raw source files often arrive as CSV.
- In this lab, the CSV intentionally includes nulls and duplicates so we can practice cleaning.

**Python file write syntax**

```python
with open(path, "w") as file:
    file.write(text)
```

**Simple example**

```python
text = "id,name\n1,Aarav"
with open("/Volumes/catalog/schema/volume/file.csv", "w") as file:
    file.write(text)
```


In [0]:
# CSV text with:
# - Null values, represented by empty fields
# - Duplicate rows for customer_id 2 and customer_id 10
csv_text = """customer_id,name,city,signup_date,total_spend,orders,loyalty_tier,email
1,Aarav,Delhi,2026-01-05,1200.50,3,Silver,aarav@example.com
2,Meera,Mumbai,2026-01-07,2500.00,5,Gold,meera@example.com
3,Kabir,Bengaluru,2026-01-08,,2,Bronze,kabir@example.com
4,Sara,Delhi,2026-01-09,875.25,,Silver,sara@example.com
5,Rohan,Chennai,2026-01-11,0,0,,rohan@example.com
6,Anika,,2026-01-12,3100.75,6,Gold,
7,Vikram,Pune,,450.00,1,Bronze,vikram@example.com
8,Nisha,Hyderabad,2026-01-14,1800.00,4,Silver,nisha@example.com
2,Meera,Mumbai,2026-01-07,2500.00,5,Gold,meera@example.com
9,Dev,Delhi,2026-01-16,999.99,2,,dev@example.com
10,Isha,Mumbai,2026-01-18,,3,Silver,isha@example.com
10,Isha,Mumbai,2026-01-18,,3,Silver,isha@example.com
"""

# In a real pipeline you would write this CSV to the Unity Catalog Volume like this:
#   with open(csv_path, "w", encoding="utf-8") as file:
#       file.write(csv_text)
#
# For this lab the CSV data is kept in the csv_text variable and will be loaded
# directly into a Spark DataFrame in the next cell (no file write needed).

data_rows = len([r for r in csv_text.strip().split("\n") if r]) - 1  # subtract header row
print(f"CSV text is defined with {data_rows} data rows (including intentional nulls and duplicates).")
print("The data will be loaded into a Spark DataFrame in the next cell.")


CSV text is defined with 12 data rows (including intentional nulls and duplicates).
The data will be loaded into a Spark DataFrame in the next cell.


### Code Explanation — Create a CSV File in a Volume

---

**`csv_text = """..."""`**
- `csv_text` — a Python variable that stores the entire CSV content as a **multi-line string**
- `"""..."""` — **triple double-quotes** define a multi-line string in Python, allowing line breaks inside without any special characters
- The first line (`customer_id,name,city,...`) is the **header row** (column names)
- Each subsequent line is one data row; empty fields (e.g., `,,`) represent **intentional null / missing values**
- Two rows are duplicated on purpose so you can practice duplicate removal later

---

**`with open(csv_path, "w", encoding="utf-8") as file:`**
- `open()` — a built-in Python function that opens a file handle for reading or writing
- `csv_path` — the full file path inside the Unity Catalog Volume (e.g., `/Volumes/catalog/schema/volume/customers.csv`)
- `"w"` — the file **mode**: `"w"` means *write*; a new file is created, or an existing one is completely overwritten
- `encoding="utf-8"` — the character encoding; UTF-8 is the universal standard that handles most languages and special characters
- `as file` — gives the opened file object the local alias `file` so you can call `.write()` on it
- `with ... as ...:` — a **context manager**; it automatically *closes* the file when the indented block finishes, even if an error occurs—no manual `file.close()` needed

---

**`file.write(csv_text)`**
- `.write()` — writes the string `csv_text` into the opened file
- After this line, the CSV file physically exists in the Unity Catalog Volume

---

**`print("CSV file created at:", csv_path)`**
- Prints a confirmation message showing the exact path where the file was saved

**Expected output**

You should see a message confirming that the CSV file was created in the Volume.


## 6. Read a CSV File with Spark

Spark can read many file formats, including CSV.

**Why this is used**
- Data engineers often load raw files into Spark DataFrames.
- Spark can infer data types from file contents.

**Function syntax**

```python
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(path)
)
```

**Simple example**

```python
df = spark.read.option("header", True).csv("/path/file.csv")
```


In [0]:
from io import StringIO
import pandas as pd

# Parse the CSV string directly into a pandas DataFrame, then convert to Spark.
# In a real pipeline you would read from a Volume file:
#   raw_df = spark.read.option("header", True).option("inferSchema", True).csv(csv_path)
#
# pandas read_csv() handles the same options (header, type inference, nulls) as Spark.
pdf = pd.read_csv(StringIO(csv_text))

# Convert the pandas DataFrame to a Spark DataFrame.
raw_df = spark.createDataFrame(pdf)

# Display the raw data.
display(raw_df)

# Show the schema.
raw_df.printSchema()


customer_id,name,city,signup_date,total_spend,orders,loyalty_tier,email
1,Aarav,Delhi,2026-01-05,1200.5,3.0,Silver,aarav@example.com
2,Meera,Mumbai,2026-01-07,2500.0,5.0,Gold,meera@example.com
3,Kabir,Bengaluru,2026-01-08,null,2.0,Bronze,kabir@example.com
4,Sara,Delhi,2026-01-09,875.25,null,Silver,sara@example.com
5,Rohan,Chennai,2026-01-11,0.0,0.0,null,rohan@example.com
6,Anika,null,2026-01-12,3100.75,6.0,Gold,null
7,Vikram,Pune,null,450.0,1.0,Bronze,vikram@example.com
8,Nisha,Hyderabad,2026-01-14,1800.0,4.0,Silver,nisha@example.com
2,Meera,Mumbai,2026-01-07,2500.0,5.0,Gold,meera@example.com
9,Dev,Delhi,2026-01-16,999.99,2.0,null,dev@example.com


root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- total_spend: double (nullable = true)
 |-- orders: double (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- email: string (nullable = true)



### Code Explanation — Read a CSV File with Spark

---

**`spark.read`**
- `spark.read` — accesses the Spark **DataFrameReader**; this is the object that knows how to load data from files, databases, and other sources

---

**`.option("header", True)`**
- `.option()` — sets a configuration option for the reader before it loads the file
- `"header", True` — tells Spark that the **first row of the CSV is the column header**, not actual data
- Without this, Spark would treat the header row as a regular data row and auto-name columns `_c0`, `_c1`, etc.

---

**`.option("inferSchema", True)`**
- `"inferSchema", True` — instructs Spark to **automatically detect the data type** of each column by scanning the file contents
- For example, a column of whole numbers becomes `IntegerType`; a column of decimals becomes `DoubleType`; text stays as `StringType`
- Setting this to `False` would read every column as a string regardless of its contents

---

**`.csv(csv_path)`**
- `.csv()` — tells the reader to load the file at `csv_path` using **CSV format**
- Returns a **Spark DataFrame** containing the file’s data distributed across the cluster

---

**`raw_df = (...)`**
- Stores the resulting DataFrame in the variable `raw_df` (*raw* signals this data has not been cleaned yet)
- The parentheses `( )` wrap the method chain purely for readability—they have no effect on how the code runs

---

**`display(raw_df)`**
- Renders the DataFrame as an interactive table in the notebook output

---

**`raw_df.printSchema()`**
- Prints column names and the data types Spark inferred for each column
- Use this to confirm numbers, dates, and strings were detected correctly before cleaning

**Expected output**

You should see 12 rows, including duplicate customer records and some missing values.


## 7. Data Profiling

**Data profiling** means checking the shape and quality of a dataset.

**Why this is used**
- To understand row counts and columns.
- To find missing values.
- To find duplicate rows.
- To check basic statistics.

**Useful function syntax**

```python
df.count()
len(df.columns)
df.describe()
df.groupBy("column").count()
```

**Simple example**

```python
row_count = df.count()
display(df.describe())
```


In [0]:
from pyspark.sql import functions as F

# Count rows and columns.
row_count = raw_df.count()
column_count = len(raw_df.columns)

print("Rows:", row_count)
print("Columns:", column_count)

# Show summary statistics for numeric and string columns.
display(raw_df.describe())

# Count how many null values exist in each column.
null_counts_df = raw_df.select([
    F.count(F.when(F.col(column).isNull(), column)).alias(column)
    for column in raw_df.columns
])

display(null_counts_df)

# Count duplicate full rows.
duplicate_row_count = raw_df.count() - raw_df.dropDuplicates().count()
print("Duplicate full rows:", duplicate_row_count)


Rows: 12
Columns: 8


summary,customer_id,name,city,signup_date,total_spend,orders,loyalty_tier,email
count,12,12,11,11,9,11,10,11
mean,5.583333333333333,null,null,null,1491.8322222222223,3.090909090909091,null,null
stddev,3.2321772378645472,null,null,null,1044.8877309163145,1.8140862964338522,null,null
min,1,Aarav,Bengaluru,2026-01-05,0.0,0.0,Bronze,aarav@example.com
max,10,Vikram,Pune,2026-01-18,3100.75,6.0,Silver,vikram@example.com


customer_id,name,city,signup_date,total_spend,orders,loyalty_tier,email
0,0,1,1,3,1,2,1


Duplicate full rows: 2


### Code Explanation — Data Profiling

---

**`from pyspark.sql import functions as F`**
- `from ... import` — imports a specific module from a library into the current notebook
- `pyspark.sql.functions` — the PySpark module containing hundreds of **built-in column functions** such as `count`, `sum`, `when`, `col`, `round`, `avg`, and many more
- `as F` — gives it the short alias `F` so you write `F.count()` instead of `pyspark.sql.functions.count()`; this is a widely used convention

---

**`row_count = raw_df.count()`**
- `.count()` — an **action** on a Spark DataFrame that triggers execution and returns the total number of rows as a Python integer
- Stored in `row_count` for printing

---

**`column_count = len(raw_df.columns)`**
- `raw_df.columns` — a Python list of all column name strings in the DataFrame
- `len()` — a built-in Python function that returns the number of items in a list; here it gives the column count

---

**`display(raw_df.describe())`**
- `.describe()` — returns a summary DataFrame with statistics (`count`, `mean`, `stddev`, `min`, `max`) for each column
- Works on both numeric and string columns
- `display()` renders the summary as a table in the notebook

---

**`raw_df.select([F.count(F.when(F.col(column).isNull(), column)).alias(column) for column in raw_df.columns])`**
- `.select([...])` — selects specific columns or expressions; here it computes one null-count expression per column
- `for column in raw_df.columns` — a Python **list comprehension** that loops over every column name and builds one expression per column
- `F.col(column)` — references a column by its string name
- `.isNull()` — returns `True` if the cell value is null, `False` otherwise
- `F.when(condition, value)` — returns `value` when the condition is `True`, otherwise returns `null`
- `F.count(...)` — counts non-null results; because `F.when` returns `null` for non-null rows, this effectively **counts only the null cells**
- `.alias(column)` — renames the result column to the original column name so the output is easy to read

---

**`duplicate_row_count = raw_df.count() - raw_df.dropDuplicates().count()`**
- `raw_df.dropDuplicates()` — returns a new DataFrame with exact duplicate rows removed
- Subtracting the two row counts gives the **number of duplicate rows** in the dataset

**Expected output**

You should see:
- Total rows and columns
- Summary statistics
- Null counts by column
- Duplicate row count


## 8. Null Handling

A **null** means a value is missing.

**Why this is used**
- Missing values can break calculations.
- Filling or removing nulls makes data more reliable.

**Common function syntax**

```python
df.fillna({"column": value})
df.dropna()
```

**Simple example**

```python
clean_df = df.fillna({"city": "Unknown", "orders": 0})
```

In this lab:
- Missing `city` becomes `"Unknown"`
- Missing `loyalty_tier` becomes `"Standard"`
- Missing `email` becomes `"not_provided@example.com"`
- Missing `orders` becomes `0`
- Missing `total_spend` becomes `0.0`


In [0]:
# Fill missing values with beginner-friendly defaults.
filled_df = raw_df.fillna({
    "city": "Unknown",
    "loyalty_tier": "Standard",
    "email": "not_provided@example.com",
    "orders": 0,
    "total_spend": 0.0
})

# Keep signup_date as null for now because invented dates can be misleading.
display(filled_df)

# Check null counts again.
display(
    filled_df.select([
        F.count(F.when(F.col(column).isNull(), column)).alias(column)
        for column in filled_df.columns
    ])
)


customer_id,name,city,signup_date,total_spend,orders,loyalty_tier,email
1,Aarav,Delhi,2026-01-05,1200.5,3.0,Silver,aarav@example.com
2,Meera,Mumbai,2026-01-07,2500.0,5.0,Gold,meera@example.com
3,Kabir,Bengaluru,2026-01-08,0.0,2.0,Bronze,kabir@example.com
4,Sara,Delhi,2026-01-09,875.25,0.0,Silver,sara@example.com
5,Rohan,Chennai,2026-01-11,0.0,0.0,Standard,rohan@example.com
6,Anika,Unknown,2026-01-12,3100.75,6.0,Gold,not_provided@example.com
7,Vikram,Pune,null,450.0,1.0,Bronze,vikram@example.com
8,Nisha,Hyderabad,2026-01-14,1800.0,4.0,Silver,nisha@example.com
2,Meera,Mumbai,2026-01-07,2500.0,5.0,Gold,meera@example.com
9,Dev,Delhi,2026-01-16,999.99,2.0,Standard,dev@example.com


customer_id,name,city,signup_date,total_spend,orders,loyalty_tier,email
0,0,0,1,0,0,0,0


### Code Explanation — Null Handling

---

**`filled_df = raw_df.fillna({...})`**
- `.fillna()` — a Spark DataFrame method that **replaces null values** with specified defaults
- Accepts a **dictionary** where each key is a column name and each value is the replacement for nulls in that column
- Returns a **new DataFrame** with nulls replaced; `raw_df` is not modified (Spark DataFrames are immutable)

---

**`"city": "Unknown"`**
- Replaces every null in the `city` column with the string `"Unknown"` so the column always has a value

---

**`"loyalty_tier": "Standard"`**
- Replaces every null in `loyalty_tier` with `"Standard"`, a safe default tier label

---

**`"email": "not_provided@example.com"`**
- Replaces missing emails with a recognizable placeholder address so the column remains populated

---

**`"orders": 0`**
- Replaces missing order counts with `0` (integer); a missing count most likely means zero orders

---

**`"total_spend": 0.0`**
- Replaces missing spend values with `0.0` (float/decimal); using `0.0` instead of `0` preserves the decimal data type

---

**`display(filled_df)`**
- Renders the filled DataFrame as a table so you can visually verify the replacements took effect

---

**`filled_df.select([F.count(F.when(F.col(column).isNull(), column)).alias(column) for column in filled_df.columns])`**
- Repeats the null-count check from the profiling step on the *filled* DataFrame
- This **confirms** that the columns you targeted now show zero nulls
- `signup_date` is expected to still show a null because no default was provided—guessing dates could introduce inaccurate data

**Expected output**

Most missing values should now be filled. `signup_date` still has a null because it is often safer not to guess dates.


## 9. Duplicate Removal

**Duplicates** are repeated records.

**Why this is used**
- Duplicate rows can inflate totals and counts.
- Removing duplicates improves data accuracy.

**Function syntax**

```python
df.dropDuplicates()
df.dropDuplicates(["column1", "column2"])
```

**Simple example**

```python
unique_df = df.dropDuplicates(["customer_id"])
```

In this lab, duplicate full rows exist for two customers. We remove exact duplicate rows.


In [0]:
# Remove exact duplicate rows.
deduped_df = filled_df.dropDuplicates()

print("Rows before duplicate removal:", filled_df.count())
print("Rows after duplicate removal:", deduped_df.count())

display(deduped_df.orderBy("customer_id"))


Rows before duplicate removal: 12
Rows after duplicate removal: 10


customer_id,name,city,signup_date,total_spend,orders,loyalty_tier,email
1,Aarav,Delhi,2026-01-05,1200.5,3.0,Silver,aarav@example.com
2,Meera,Mumbai,2026-01-07,2500.0,5.0,Gold,meera@example.com
3,Kabir,Bengaluru,2026-01-08,0.0,2.0,Bronze,kabir@example.com
4,Sara,Delhi,2026-01-09,875.25,0.0,Silver,sara@example.com
5,Rohan,Chennai,2026-01-11,0.0,0.0,Standard,rohan@example.com
6,Anika,Unknown,2026-01-12,3100.75,6.0,Gold,not_provided@example.com
7,Vikram,Pune,null,450.0,1.0,Bronze,vikram@example.com
8,Nisha,Hyderabad,2026-01-14,1800.0,4.0,Silver,nisha@example.com
9,Dev,Delhi,2026-01-16,999.99,2.0,Standard,dev@example.com
10,Isha,Mumbai,2026-01-18,0.0,3.0,Silver,isha@example.com


### Code Explanation — Duplicate Removal

---

**`deduped_df = filled_df.dropDuplicates()`**
- `.dropDuplicates()` — returns a **new DataFrame with exact duplicate rows removed**
- A row is considered a duplicate only if **every single column value** matches another row exactly
- No arguments are passed, so all columns are compared
- `filled_df` is not changed; `deduped_df` is a new deduplicated DataFrame

---

**`print("Rows before duplicate removal:", filled_df.count())`**
- Calls `.count()` on the *filled* DataFrame to get the row count **before** deduplication
- Prints it so you can see the comparison

---

**`print("Rows after duplicate removal:", deduped_df.count())`**
- Calls `.count()` on the *deduplicated* DataFrame to show how many rows remain after removing duplicates

---

**`display(deduped_df.orderBy("customer_id"))`**
- `.orderBy("customer_id")` — **sorts** the DataFrame by the `customer_id` column in ascending order (smallest ID first, which is the default)
- `display()` renders the sorted result as a table
- Sorting by ID makes it easy to visually verify which duplicate customer IDs were removed

**Expected output**

The row count should decrease from 12 to 10 because two exact duplicate rows are removed.


## 10. Transformations

A **transformation** creates a new DataFrame from an existing DataFrame.

Transformations are not actions by themselves. Spark builds a plan and runs it when an action such as `display()`, `count()`, or `write` is called.

**Why this is used**
- To create new columns.
- To rename or format columns.
- To prepare data for analytics.

**Common function syntax**

```python
df.withColumn("new_column", expression)
df.select("column1", "column2")
df.orderBy("column")
```

**Simple example**

```python
df2 = df.withColumn("total_with_tax", F.col("total") * 1.18)
```


In [0]:
# Create transformed columns:
# - signup_date_as_date converts text to a DateType.
# - average_order_value divides total spend by orders.
# - customer_segment labels customers by spend.
transformed_df = (
    deduped_df
    .withColumn("signup_date_as_date", F.to_date(F.col("signup_date")))
    .withColumn(
        "average_order_value",
        F.when(F.col("orders") > 0, F.round(F.col("total_spend") / F.col("orders"), 2))
         .otherwise(F.lit(0.0))
    )
    .withColumn(
        "customer_segment",
        F.when(F.col("total_spend") >= 2500, F.lit("High Value"))
         .when(F.col("total_spend") >= 1000, F.lit("Medium Value"))
         .otherwise(F.lit("Low Value"))
    )
)

display(transformed_df.orderBy("customer_id"))


customer_id,name,city,signup_date,total_spend,orders,loyalty_tier,email,signup_date_as_date,average_order_value,customer_segment
1,Aarav,Delhi,2026-01-05,1200.5,3.0,Silver,aarav@example.com,2026-01-05,400.17,Medium Value
2,Meera,Mumbai,2026-01-07,2500.0,5.0,Gold,meera@example.com,2026-01-07,500.0,High Value
3,Kabir,Bengaluru,2026-01-08,0.0,2.0,Bronze,kabir@example.com,2026-01-08,0.0,Low Value
4,Sara,Delhi,2026-01-09,875.25,0.0,Silver,sara@example.com,2026-01-09,0.0,Low Value
5,Rohan,Chennai,2026-01-11,0.0,0.0,Standard,rohan@example.com,2026-01-11,0.0,Low Value
6,Anika,Unknown,2026-01-12,3100.75,6.0,Gold,not_provided@example.com,2026-01-12,516.79,High Value
7,Vikram,Pune,null,450.0,1.0,Bronze,vikram@example.com,null,450.0,Low Value
8,Nisha,Hyderabad,2026-01-14,1800.0,4.0,Silver,nisha@example.com,2026-01-14,450.0,Medium Value
9,Dev,Delhi,2026-01-16,999.99,2.0,Standard,dev@example.com,2026-01-16,500.0,Low Value
10,Isha,Mumbai,2026-01-18,0.0,3.0,Silver,isha@example.com,2026-01-18,0.0,Low Value


### Code Explanation — Transformations

---

**`transformed_df = (deduped_df ...)`**
- The outer parentheses `( )` wrap a **method chain**, allowing multiple `.withColumn()` calls to be written on separate lines for readability
- Each `.withColumn()` adds a new column to the DataFrame without changing any existing ones
- The result of the entire chain is stored in `transformed_df`

---

**`.withColumn("signup_date_as_date", F.to_date(F.col("signup_date")))`**
- `.withColumn(name, expression)` — adds a new column named `name`, computed from `expression`
- `F.col("signup_date")` — references the existing `signup_date` column by its string name
- `F.to_date()` — converts a **string column into a proper DateType column**
- This matters because dates stored as strings cannot be used in date calculations (e.g., date differences, range filters)

---

**`.withColumn("average_order_value", F.when(F.col("orders") > 0, F.round(...)).otherwise(F.lit(0.0)))`**
- `F.when(condition, value)` — like an `if` statement; returns `value` when the condition is `True`
- `F.col("orders") > 0` — the condition; checks that `orders` is greater than zero to **avoid division by zero**
- `F.round(F.col("total_spend") / F.col("orders"), 2)` — divides total spend by orders and rounds the result to 2 decimal places
- `.otherwise(F.lit(0.0))` — the `else` branch; returns `0.0` when `orders` is zero
- `F.lit(0.0)` — creates a **literal constant** column with the fixed value `0.0`

---

**`.withColumn("customer_segment", F.when(...).when(...).otherwise(...))`**
- Chaining multiple `.when()` calls creates **if / else-if / else** logic:
  - First `.when`: if `total_spend >= 2500` → label as `"High Value"`
  - Second `.when`: if `total_spend >= 1000` → label as `"Medium Value"`
  - `.otherwise(F.lit("Low Value"))` → any customer not matching the above becomes `"Low Value"`

---

**`display(transformed_df.orderBy("customer_id"))`**
- Displays the result sorted by `customer_id` so you can review the new columns row by row in order

**Expected output**

You should see new columns for converted date, average order value, and customer segment.


## 11. Filtering

**Filtering** keeps only rows that match a condition.

**Why this is used**
- To focus on a subset of data.
- To answer questions such as "Which customers spent more than 1000?"

**Function syntax**

```python
df.filter(condition)
```

**Simple example**

```python
high_spend_df = df.filter(F.col("total_spend") > 1000)
```


In [0]:
# Keep customers who spent more than or equal to 1000.
medium_or_high_value_df = transformed_df.filter(F.col("total_spend") >= 1000)

display(
    medium_or_high_value_df.select(
        "customer_id",
        "name",
        "city",
        "total_spend",
        "customer_segment"
    ).orderBy(F.col("total_spend").desc())
)


customer_id,name,city,total_spend,customer_segment
6,Anika,Unknown,3100.75,High Value
2,Meera,Mumbai,2500.0,High Value
8,Nisha,Hyderabad,1800.0,Medium Value
1,Aarav,Delhi,1200.5,Medium Value


### Code Explanation — Filtering

---

**`medium_or_high_value_df = transformed_df.filter(F.col("total_spend") >= 1000)`**
- `.filter()` — keeps only the rows where the given condition evaluates to `True`; all other rows are discarded
- `F.col("total_spend")` — references the `total_spend` column
- `>= 1000` — the condition: total spend must be **greater than or equal to** 1000
- The result is a new DataFrame stored in `medium_or_high_value_df`
- `.filter()` is a **transformation** (lazy): Spark records the plan but does not actually scan data until an action like `display()` or `count()` is called

---

**`.select("customer_id", "name", "city", "total_spend", "customer_segment")`**
- `.select()` — picks only the listed columns to display
- Selecting specific columns (instead of showing all) keeps the output focused and easier to read

---

**`.orderBy(F.col("total_spend").desc())`**
- `.orderBy()` — sorts the result rows by the specified column
- `F.col("total_spend")` — references the column to sort by
- `.desc()` — a column method that signals **descending order** (highest value first)
- Use `.asc()` or omit the method entirely for ascending (lowest first, which is the default)

**Expected output**

You should see only customers with `total_spend` greater than or equal to 1000.


## 12. Aggregations

**Aggregation** summarizes many rows into fewer rows.

**Why this is used**
- To calculate totals, averages, counts, minimums, and maximums.
- To create reports by group, such as spend by city or tier.

**Function syntax**

```python
df.groupBy("column").agg(F.sum("number_column"))
```

**Simple example**

```python
city_summary = df.groupBy("city").agg(F.sum("total_spend").alias("total_spend"))
```


In [0]:
# Summarize customer activity by city.
city_summary_df = (
    transformed_df
    .groupBy("city")
    .agg(
        F.count("*").alias("customer_count"),
        F.round(F.sum("total_spend"), 2).alias("total_spend"),
        F.sum("orders").alias("total_orders"),
        F.round(F.avg("average_order_value"), 2).alias("avg_order_value")
    )
    .orderBy(F.col("total_spend").desc())
)

display(city_summary_df)

# Summarize customer activity by loyalty tier.
tier_summary_df = (
    transformed_df
    .groupBy("loyalty_tier")
    .agg(
        F.count("*").alias("customer_count"),
        F.round(F.sum("total_spend"), 2).alias("total_spend"),
        F.round(F.avg("total_spend"), 2).alias("avg_customer_spend")
    )
    .orderBy(F.col("total_spend").desc())
)

display(tier_summary_df)


city,customer_count,total_spend,total_orders,avg_order_value
Unknown,1,3100.75,6.0,516.79
Delhi,3,3075.74,5.0,300.06
Mumbai,2,2500.0,8.0,250.0
Hyderabad,1,1800.0,4.0,450.0
Pune,1,450.0,1.0,450.0
Bengaluru,1,0.0,2.0,0.0
Chennai,1,0.0,0.0,0.0


loyalty_tier,customer_count,total_spend,avg_customer_spend
Gold,2,5600.75,2800.38
Silver,4,3875.75,968.94
Standard,2,999.99,500.0
Bronze,2,450.0,225.0


### Code Explanation — Aggregations

---

**`city_summary_df = transformed_df.groupBy("city").agg(...).orderBy(...)`**
- `.groupBy("city")` — groups all rows that share the same value in the `city` column; equivalent to `GROUP BY city` in SQL
- After grouping, aggregate functions are applied **per group** (per city)

---

**`.agg(F.count("*").alias("customer_count"), ...)`**
- `.agg()` — applies one or more **aggregate functions** to each group in one step
- `F.count("*")` — counts **all rows** in each group; equivalent to `COUNT(*)` in SQL
- `.alias("customer_count")` — renames the resulting column to `"customer_count"` so it is readable

---

**`F.round(F.sum("total_spend"), 2).alias("total_spend")`**
- `F.sum("total_spend")` — adds up all `total_spend` values within each city group
- `F.round(..., 2)` — rounds the sum to **2 decimal places**

---

**`F.sum("orders").alias("total_orders")`**
- Sums the `orders` column for each city group to produce total orders placed per city

---

**`F.round(F.avg("average_order_value"), 2).alias("avg_order_value")`**
- `F.avg()` — computes the **arithmetic mean** (average) of a column within each group
- Shows the typical order size for customers in each city

---

**`.orderBy(F.col("total_spend").desc())`**
- Sorts the grouped results by `total_spend` descending so the **highest-spending city appears first**

---

**`tier_summary_df = transformed_df.groupBy("loyalty_tier").agg(...)`**
- Same grouping and aggregation pattern, but groups by `loyalty_tier` instead of `city`
- `F.avg("total_spend")` — computes **average spend per loyalty tier** to see which tier spends more on average
- `F.count("*")` — counts how many customers belong to each tier

**Expected output**

You should see grouped summaries by city and by loyalty tier.


## 13. Delta Tables

**Delta Lake** is the storage layer commonly used with Databricks tables.

**Why this is used**
- Delta tables support reliable writes.
- They support schema management.
- They support updates, deletes, and time travel.
- They are a strong default for lakehouse data.

**Write syntax**

```python
df.write.format("delta").mode("overwrite").saveAsTable("catalog.schema.table")
```

**Read syntax**

```python
table_df = spark.table("catalog.schema.table")
```

**Simple example**

```python
df.write.format("delta").mode("overwrite").saveAsTable("my_catalog.my_schema.my_table")
```


In [0]:
# Define table names.
bronze_table = f"{full_schema_name}.bronze_customers"
silver_city_table = f"{full_schema_name}.silver_city_summary"
silver_tier_table = f"{full_schema_name}.silver_tier_summary"

# Write the cleaned customer data as a Delta table.
(
    transformed_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

print("Bronze Delta table created:", bronze_table)

# Read the Delta table back.
bronze_df = spark.table(bronze_table)
display(bronze_df.orderBy("customer_id"))


Bronze Delta table created: beginner_catalog.databricks_basics.bronze_customers


customer_id,name,city,signup_date,total_spend,orders,loyalty_tier,email,signup_date_as_date,average_order_value,customer_segment
1,Aarav,Delhi,2026-01-05,1200.5,3,Silver,aarav@example.com,2026-01-05,400.17,Medium Value
2,Meera,Mumbai,2026-01-07,2500.0,5,Gold,meera@example.com,2026-01-07,500.0,High Value
3,Kabir,Bengaluru,2026-01-08,0.0,2,Bronze,kabir@example.com,2026-01-08,0.0,Low Value
4,Sara,Delhi,2026-01-09,875.25,0,Silver,sara@example.com,2026-01-09,0.0,Low Value
5,Rohan,Chennai,2026-01-11,0.0,0,Standard,rohan@example.com,2026-01-11,0.0,Low Value
6,Anika,Unknown,2026-01-12,3100.75,6,Gold,not_provided@example.com,2026-01-12,516.79,High Value
7,Vikram,Pune,null,450.0,1,Bronze,vikram@example.com,null,450.0,Low Value
8,Nisha,Hyderabad,2026-01-14,1800.0,4,Silver,nisha@example.com,2026-01-14,450.0,Medium Value
9,Dev,Delhi,2026-01-16,999.99,2,Standard,dev@example.com,2026-01-16,500.0,Low Value
10,Isha,Mumbai,2026-01-18,0.0,3,Silver,isha@example.com,2026-01-18,0.0,Low Value


### Code Explanation — Writing the Bronze Delta Table

---

**`bronze_table = f"{full_schema_name}.bronze_customers"`**
- Builds the **fully qualified table name** by combining the schema variable with the table name `bronze_customers`
- Using an f-string means you only need to change `full_schema_name` in one place if the schema ever changes

---

**`transformed_df.write`**
- `.write` — accesses the Spark **DataFrameWriter**, the object that saves DataFrames to storage

---

**`.format("delta")`**
- `.format("delta")` — specifies **Delta Lake** as the storage format
- Delta adds ACID transactions, schema enforcement, and time travel (version history) on top of Parquet files
- Delta is the default and recommended format in Databricks

---

**`.mode("overwrite")`**
- `.mode("overwrite")` — if the table already exists, its data is **completely replaced**
- Other modes: `"append"` adds new rows, `"ignore"` skips if the table exists, `"error"` (default) raises an error if it exists

---

**`.option("overwriteSchema", "true")`**
- Allows the **table schema (column structure) to also be replaced** along with the data
- Without this option, Delta raises an error if the new DataFrame has different columns from the existing table

---

**`.saveAsTable(bronze_table)`**
- `.saveAsTable()` — saves the DataFrame as a **managed Delta table** registered in Unity Catalog
- A *managed table* means Unity Catalog controls both the metadata and the underlying data files

---

**`bronze_df = spark.table(bronze_table)`**
- `spark.table()` — reads an existing Delta table back into a DataFrame by its fully qualified name
- Used here as a **read-back verification** to confirm the write succeeded

---

**`display(bronze_df.orderBy("customer_id"))`**
- Displays the saved table data sorted by `customer_id` so you can visually verify the contents are correct

**Expected output**

You should see a Delta table named `bronze_customers` in your Unity Catalog schema.


## 14. Bronze and Silver Layers

A lakehouse often organizes data into layers.

### Bronze
Bronze data is close to the raw source data. It may be lightly cleaned.

### Silver
Silver data is cleaned, transformed, and ready for analytics.

**Why this is used**
- Layers make data pipelines easier to understand.
- Bronze keeps source-level detail.
- Silver prepares reliable business views.

**Simple pattern**

```text
Raw file -> Bronze table -> Silver table
```

In this notebook:
- `bronze_customers` stores cleaned customer-level data.
- `silver_city_summary` stores city-level analytics.
- `silver_tier_summary` stores loyalty-tier analytics.


In [0]:
# Write the city summary as a Silver Delta table.
(
    city_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_city_table)
)

# Write the loyalty tier summary as a Silver Delta table.
(
    tier_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_tier_table)
)

print("Silver Delta tables created:")
print("-", silver_city_table)
print("-", silver_tier_table)

display(spark.table(silver_city_table))
display(spark.table(silver_tier_table))


Silver Delta tables created:
- beginner_catalog.databricks_basics.silver_city_summary
- beginner_catalog.databricks_basics.silver_tier_summary


city,customer_count,total_spend,total_orders,avg_order_value
Unknown,1,3100.75,6,516.79
Delhi,3,3075.74,5,300.06
Mumbai,2,2500.0,8,250.0
Hyderabad,1,1800.0,4,450.0
Pune,1,450.0,1,450.0
Chennai,1,0.0,0,0.0
Bengaluru,1,0.0,2,0.0


loyalty_tier,customer_count,total_spend,avg_customer_spend
Gold,2,5600.75,2800.38
Silver,4,3875.75,968.94
Standard,2,999.99,500.0
Bronze,2,450.0,225.0


### Code Explanation — Writing Silver Delta Tables

---

**`city_summary_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_city_table)`**
- The same write pattern used for the bronze table is applied here
- `city_summary_df` — the aggregated city-level summary DataFrame built in the aggregations step
- `silver_city_table` — the fully qualified name for the Silver-layer city summary table
- `.format("delta")` — writes as a Delta table
- `.mode("overwrite")` — replaces any existing data
- `.option("overwriteSchema", "true")` — allows the column structure to be updated
- `.saveAsTable(silver_city_table)` — registers the table in Unity Catalog

---

**`tier_summary_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_tier_table)`**
- Writes the loyalty-tier summary as a second, separate Silver Delta table
- `silver_tier_table` — the fully qualified name for the Silver-layer tier summary table

---

**`print("Silver Delta tables created:")` / `print("-", silver_city_table)` / `print("-", silver_tier_table)`**
- Print confirmation messages listing the exact names of the tables just created
- `"-"` before each name acts as a bullet point in the output, making it easier to read

---

**`display(spark.table(silver_city_table))`**
- `spark.table()` — reads the `silver_city_summary` table directly from Unity Catalog into a DataFrame
- `display()` — renders it as a table; this is a **read-back verification** to confirm the data was saved correctly

---

**`display(spark.table(silver_tier_table))`**
- Same read-back verification for the tier summary table

**Expected output**

You should see two Silver tables containing summary results that are easier to use for reports and dashboards.


## 15. SQL Analytics

Databricks notebooks can run SQL cells using `%sql`.

**Why this is used**
- SQL is widely used by analysts and data engineers.
- SQL is easy for querying tables.
- SQL can filter, group, sort, and join data.

**SQL syntax**

```sql
SELECT column1, column2
FROM catalog.schema.table
WHERE condition
ORDER BY column1;
```

**Simple example**

```sql
SELECT city, SUM(total_spend)
FROM beginner_catalog.databricks_basics.bronze_customers
GROUP BY city;
```


In [0]:
%sql
USE CATALOG ${catalog_name};
USE SCHEMA ${schema_name};


### Code Explanation — Setting the Active Catalog and Schema in SQL

---

**`%sql`**
- A **magic command** at the very top of the cell that switches the cell language to **SQL**
- Without `%sql`, Databricks would try to run the cell as Python (the notebook’s default language)
- Each SQL cell in a Python notebook must start with `%sql`

---

**`USE CATALOG ${catalog_name};`**
- `USE CATALOG` — a SQL command that sets the **active catalog** for this SQL session
- `${catalog_name}` — the `${}` syntax references a Databricks **notebook widget or parameter** named `catalog_name` that was defined earlier
- After this command, all SQL queries in this session can omit the catalog prefix

---

**`USE SCHEMA ${schema_name};`**
- `USE SCHEMA` — sets the **active schema (database)** for this SQL session
- After this, you can write `SELECT * FROM bronze_customers` instead of the full path `SELECT * FROM catalog.schema.bronze_customers`
- The `;` at the end of each SQL statement is optional in Databricks but is good practice for clarity

### SQL Query 1: View Bronze Customer Data

This query reads customer-level data from the Bronze table.

**Syntax**

```sql
SELECT *
FROM table_name
LIMIT number;
```


In [0]:
%sql
SELECT *
FROM bronze_customers
ORDER BY customer_id;


customer_id,name,city,signup_date,total_spend,orders,loyalty_tier,email,signup_date_as_date,average_order_value,customer_segment
1,Aarav,Delhi,2026-01-05,1200.5,3,Silver,aarav@example.com,2026-01-05,400.17,Medium Value
2,Meera,Mumbai,2026-01-07,2500.0,5,Gold,meera@example.com,2026-01-07,500.0,High Value
3,Kabir,Bengaluru,2026-01-08,0.0,2,Bronze,kabir@example.com,2026-01-08,0.0,Low Value
4,Sara,Delhi,2026-01-09,875.25,0,Silver,sara@example.com,2026-01-09,0.0,Low Value
5,Rohan,Chennai,2026-01-11,0.0,0,Standard,rohan@example.com,2026-01-11,0.0,Low Value
6,Anika,Unknown,2026-01-12,3100.75,6,Gold,not_provided@example.com,2026-01-12,516.79,High Value
7,Vikram,Pune,null,450.0,1,Bronze,vikram@example.com,null,450.0,Low Value
8,Nisha,Hyderabad,2026-01-14,1800.0,4,Silver,nisha@example.com,2026-01-14,450.0,Medium Value
9,Dev,Delhi,2026-01-16,999.99,2,Standard,dev@example.com,2026-01-16,500.0,Low Value
10,Isha,Mumbai,2026-01-18,0.0,3,Silver,isha@example.com,2026-01-18,0.0,Low Value


### Code Explanation — SQL Query 1: View Bronze Customer Data

---

**`%sql`**
- Magic command that tells Databricks to run this cell as SQL

---

**`SELECT *`**
- `SELECT` — the SQL keyword that specifies **which columns to retrieve**
- `*` — a **wildcard** meaning “all columns”; returns every column that exists in the table

---

**`FROM bronze_customers`**
- `FROM` — specifies **which table** to query
- `bronze_customers` — the table name; because `USE CATALOG` and `USE SCHEMA` were set in the previous cell, there is no need to write the full `catalog.schema.bronze_customers` path

---

**`ORDER BY customer_id;`**
- `ORDER BY` — **sorts** the returned rows by the specified column
- `customer_id` — sorts in **ascending order** by default (lowest ID first)
- `;` — ends the SQL statement

**Expected output**

You should see cleaned customer records from the Bronze Delta table.


### SQL Query 2: Customers by Segment

This query counts customers in each segment.

**Syntax**

```sql
SELECT group_column, COUNT(*) AS count_name
FROM table_name
GROUP BY group_column;
```


In [0]:
%sql
SELECT
  customer_segment,
  COUNT(*) AS customer_count,
  ROUND(SUM(total_spend), 2) AS total_spend
FROM bronze_customers
GROUP BY customer_segment
ORDER BY total_spend DESC;


customer_segment,customer_count,total_spend
High Value,2,5600.75
Medium Value,2,3000.5
Low Value,6,2325.24


### Code Explanation — SQL Query 2: Customers by Segment

---

**`SELECT customer_segment, COUNT(*) AS customer_count, ROUND(SUM(total_spend), 2) AS total_spend`**
- `SELECT` — lists the columns and expressions to return
- `customer_segment` — the column we are grouping by; it will appear as one row per unique segment value
- `COUNT(*) AS customer_count` — `COUNT(*)` counts **all rows** in each group; `AS customer_count` renames the result column
- `SUM(total_spend)` — adds up all `total_spend` values within each segment group
- `ROUND(..., 2)` — rounds the sum to **2 decimal places**
- `AS total_spend` — renames the rounded sum column

---

**`FROM bronze_customers`**
- Reads data from the bronze Delta table

---

**`GROUP BY customer_segment`**
- Collapses all rows with the same `customer_segment` value into a **single summary row**
- Required in SQL whenever aggregate functions (`COUNT`, `SUM`, `AVG`, etc.) are used alongside non-aggregated columns

---

**`ORDER BY total_spend DESC;`**
- `ORDER BY total_spend` — sorts the grouped results by the `total_spend` column
- `DESC` — **descending order** (highest spend first)
- Without `DESC`, results would sort ascending (lowest first)

**Expected output**

You should see customer counts and total spend by customer segment.


### SQL Query 3: Analyze Silver City Summary

This query reads from a Silver summary table.

**Syntax**

```sql
SELECT *
FROM silver_table
ORDER BY metric DESC;
```


In [0]:
%sql
SELECT
  city,
  customer_count,
  total_spend,
  total_orders,
  avg_order_value
FROM silver_city_summary
ORDER BY total_spend DESC;


city,customer_count,total_spend,total_orders,avg_order_value
Unknown,1,3100.75,6,516.79
Delhi,3,3075.74,5,300.06
Mumbai,2,2500.0,8,250.0
Hyderabad,1,1800.0,4,450.0
Pune,1,450.0,1,450.0
Chennai,1,0.0,0,0.0
Bengaluru,1,0.0,2,0.0


### Code Explanation — SQL Query 3: Silver City Summary

---

**`SELECT city, customer_count, total_spend, total_orders, avg_order_value`**
- Lists the **exact columns** to retrieve from the Silver table
- Naming specific columns (instead of `SELECT *`) is a best practice: it makes the query’s intent clear and prevents unexpected columns from appearing if the table schema changes

---

**`FROM silver_city_summary`**
- Reads from the `silver_city_summary` Delta table
- This table already contains **pre-aggregated, city-level data** written in the Silver layer step

---

**`ORDER BY total_spend DESC;`**
- Sorts cities by `total_spend` in **descending order** so the highest-spending cities appear at the top
- Because the Silver table is already one row per city (pre-aggregated), **no `GROUP BY` is needed** here—each row already represents one city

**Expected output**

You should see cities ordered by total spend.


### SQL Query 4: Find Customers Missing Signup Dates

This query checks rows where the signup date is missing.

**Syntax**

```sql
SELECT *
FROM table_name
WHERE column_name IS NULL;
```


In [0]:
%sql
SELECT
  customer_id,
  name,
  city,
  signup_date,
  total_spend
FROM bronze_customers
WHERE signup_date IS NULL;


customer_id,name,city,signup_date,total_spend
7,Vikram,Pune,null,450.0


### Code Explanation — SQL Query 4: Find Customers with Missing Signup Dates

---

**`SELECT customer_id, name, city, signup_date, total_spend`**
- Selects the columns most relevant to identifying customers with missing signup dates
- Choosing specific columns keeps the result focused rather than showing all columns

---

**`FROM bronze_customers`**
- Reads from the bronze customer Delta table

---

**`WHERE signup_date IS NULL;`**
- `WHERE` — filters rows; **only rows where the condition evaluates to `True` are returned**
- `signup_date IS NULL` — the SQL condition that checks for a missing value
- **Important:** in SQL you must use `IS NULL` (not `= NULL`) to test for nulls
  - This is because `NULL = NULL` evaluates to *unknown* (not `True`) in SQL logic
  - `IS NULL` is the correct, standard way to check for missing values

**Expected output**

You should see customers where the original `signup_date` was missing.


## 16. Optional Cleanup

Cleanup removes objects created during practice.

**Why this is used**
- To avoid clutter.
- To keep lab environments clean.

**SQL syntax**

```sql
DROP TABLE IF EXISTS table_name;
DROP VOLUME IF EXISTS volume_name;
DROP SCHEMA IF EXISTS schema_name CASCADE;
DROP CATALOG IF EXISTS catalog_name CASCADE;
```

The cleanup commands are commented out so you do not accidentally delete your lab work.


In [0]:
# Uncomment these lines only when you are sure you want to delete the lab objects.
#
# spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")
# spark.sql(f"DROP TABLE IF EXISTS {silver_city_table}")
# spark.sql(f"DROP TABLE IF EXISTS {silver_tier_table}")
# spark.sql(f"DROP VOLUME IF EXISTS {full_volume_name}")
# spark.sql(f"DROP SCHEMA IF EXISTS {full_schema_name} CASCADE")
# spark.sql(f"DROP CATALOG IF EXISTS {catalog_name} CASCADE")
#
# print("Cleanup complete.")


### Code Explanation — Optional Cleanup

---

**`# spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")`**
- Lines starting with `#` are **comments**; Python completely ignores them at runtime
- `DROP TABLE IF EXISTS` — a SQL command that **permanently deletes** a Delta table from Unity Catalog
- `IF EXISTS` — prevents an error if the table does not exist
- This line is **commented out** intentionally so it does not run accidentally; uncomment by removing the `#` only when you truly want to delete

---

**`# spark.sql(f"DROP TABLE IF EXISTS {silver_city_table}")` / `# spark.sql(f"DROP TABLE IF EXISTS {silver_tier_table}")`**
- Same pattern to delete the two Silver summary tables

---

**`# spark.sql(f"DROP VOLUME IF EXISTS {full_volume_name}")`**
- `DROP VOLUME IF EXISTS` — removes the Unity Catalog Volume **and all files stored inside it** (including the CSV file created earlier)

---

**`# spark.sql(f"DROP SCHEMA IF EXISTS {full_schema_name} CASCADE")`**
- `DROP SCHEMA ... CASCADE` — deletes the schema and **everything inside it** (tables, volumes, and their data)
- `CASCADE` is required because the schema is not empty—without it, the command would fail

---

**`# spark.sql(f"DROP CATALOG IF EXISTS {catalog_name} CASCADE")`**
- `DROP CATALOG ... CASCADE` — deletes the **entire catalog** including all schemas, tables, and volumes inside it
- Use with extreme caution; this operation is **irreversible**

---

**`# print("Cleanup complete.")`**
- Would print a confirmation message once all cleanup steps have run successfully

## Key Takeaways

You have completed a beginner-friendly Databricks lakehouse workflow.

Remember these ideas:

1. **Python** helps you write logic, variables, lists, and loops.
2. **Spark DataFrames** store data in rows and columns and can process large datasets.
3. **Unity Catalog** organizes and governs data assets.
4. **Catalogs, schemas, and volumes** help structure tables and files.
5. **CSV files** are common raw data sources.
6. **Data profiling** helps you understand data quality before cleaning.
7. **Null handling** fills or manages missing values.
8. **Duplicate removal** prevents repeated records from affecting results.
9. **Transformations** create new columns and prepare data for analysis.
10. **Filtering** keeps only the rows you need.
11. **Aggregations** summarize data for reporting.
12. **Delta tables** are reliable Databricks tables for lakehouse workloads.
13. **Bronze and Silver layers** make pipelines easier to understand and maintain.
14. **SQL analytics** lets you query Delta tables using familiar SQL.

You now have a complete mini project:

```text
Create CSV in Volume
Read CSV with Spark
Profile data
Clean nulls and duplicates
Transform and filter data
Write Bronze Delta table
Create Silver summary tables
Query tables with SQL
```
